In [ ]:
!pip install pandas==2.2.3
!pip install scikit-learn==1.6.0
!pip install matplotlib==3.9.3
!pip install kagglehub[pandas-datasets]

In [ ]:
# Import the libraries
from __future__ import print_function
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
from sklearn.svm import LinearSVC

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Download latest version
file_path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

raw_data = pd.read_csv(file_path + "/creditcard.csv")

raw_data.head()

In [ ]:
# EDA
labels = raw_data.Class.unique() # Number of classes

sizes = raw_data.Class.value_counts().values # Count of each class

# Plot the class value counts
fig, ax = plt.subplots()
ax.pie(sizes, labels=labels, autopct='%1.3f%%')
ax.set_title('Target Variable Value Counts')
plt.show()

#Which features affect the result the most
correlation_values = raw_data.corr()['Class'].drop('Class')
correlation_values.plot(kind='barh', figsize=(10, 6))

In [ ]:
'''
The above dataset is highly imbalanced. We will use decision trees to classify the data.
If we use a Deciosion Tree Classifier we will have to use custom class weights to balance the dataset.

Given below is the code to implement Decision Tree Classifier with balanced class weights.
w_train = compute_sample_weight('balanced', y_train)
dt = DecisionTreeClassifier(max_depth=4, random_state=35)
dt.fit(X_train, y_train, sample_weight=w_train)

Incomparison, we can use SVM with built-in class weight balancing.
'''
#DATA PREPROCESSING
# standardize features by removing the mean and scaling to unit variance
raw_data.iloc[:, 1:30] = StandardScaler().fit_transform(raw_data.iloc[:, 1:30])
data_matrix = raw_data.values

# X: feature matrix (for this analysis, we exclude the Time variable from the dataset)
X = data_matrix[:, 1:30]

# y: labels vector
y = data_matrix[:, 30]

# data normalization
X = normalize(X, norm="l1")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
#BUILDING THE MODEL
svm = LinearSVC(class_weight='balanced', random_state=31, loss="hinge", fit_intercept=False)
svm.fit(X_train, y_train)

In [ ]:
#MODEL EVALUATION
y_pred_svm = svm.decision_function(X_test)
roc_auc_svm = roc_auc_score(y_test, y_pred_svm)
print("SVM ROC-AUC score: {0:.3f}".format(roc_auc_svm))